# Lab 4: Graph-Enriched Retrieval

In this lab, you will add a Cypher `retrieval_query` to `Neo4jContextProvider` that traverses graph relationships after vector search, giving the agent structured metadata like genres, actors, and directors alongside plot text. Basic vector search only returns the text that was embedded — graph enrichment walks the relationships around each match to collect connected data the LLM can reference.

## What you will learn

- The limitation of basic vector search (plot text only, no metadata)
- How to write a `retrieval_query` that receives `node` and `score` from the index search
- How to traverse `IN_GENRE`, `ACTED_IN`, and `DIRECTED` relationships
- How graph enrichment produces richer, more specific agent responses

## Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

Environment loaded successfully


## Import Dependencies

In [2]:
from llm_provider import get_client, get_embedder
from agent_framework_neo4j import Neo4jContextProvider, Neo4jSettings

## Load Settings and Create Embedder

In [3]:
neo4j_settings = Neo4jSettings()
embedder = get_embedder()

## Define a Retrieval Query

The `retrieval_query` runs after vector search to traverse graph relationships from each matched node, enriching plain text results with structured metadata that the LLM can reference. It receives two variables:

- **`node`** — The matched `Movie` node from the index
- **`score`** — The relevance score from the search

Write a Cypher query that traverses outward from the matched node to collect:
- Genres via `(node)-[:IN_GENRE]->(g:Genre)`
- Up to 5 actors via `(p:Person)-[:ACTED_IN]->(node)`
- Directors via `(d:Person)-[:DIRECTED]->(node)`

Return `node.plot AS text`, `score`, `node.title AS title`, `node.released AS released`, plus the collected genres, actors, and directors.

In [4]:
RETRIEVAL_QUERY = """
MATCH (node)-[:IN_GENRE]->(g:Genre)
WITH node, score, collect(DISTINCT g.name) AS genres
OPTIONAL MATCH (p:Person)-[:ACTED_IN]->(node)
// [0..5] is Cypher list slicing — it takes elements from index 0 up to (but not including) 5,
// limiting the result to at most 5 actors per movie
WITH node, score, genres, collect(DISTINCT p.name)[0..5] AS actors
OPTIONAL MATCH (d:Person)-[:DIRECTED]->(node)
WITH node, score, genres, actors, collect(DISTINCT d.name) AS directors
WHERE score IS NOT NULL
RETURN
    node.plot AS text,
    score,
    node.title AS title,
    node.released AS released,
    genres,
    actors,
    directors
ORDER BY score DESC
"""

## Create the Graph-Enriched Provider

Create a `Neo4jContextProvider` with `index_type="vector"` and pass the `retrieval_query` parameter. Passing `retrieval_query` switches the provider from returning just text and a score to returning the full set of fields your Cypher query specifies — in this case, title, year, genres, actors, and directors alongside the plot text.

In [5]:
provider = Neo4jContextProvider(
    uri=neo4j_settings.uri,
    username=neo4j_settings.username,
    password=neo4j_settings.get_password(),
    index_name=neo4j_settings.vector_index_name,
    index_type="vector",
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    top_k=5,
    context_prompt=(
        "## Graph-Enriched Movie Context\n"
        "The following information combines semantic search results with "
        "graph traversal to provide movie, actor, genre, and director context:"
    ),
)

## Run the Agent

The agent now receives context that includes plot text, movie title, release year, genres, actors, and directors for each match.

In [6]:
async with provider:
    client = get_client()

    agent = client.as_agent(
        name="graph-enriched-agent",
        instructions=(
            "You are a helpful assistant that answers questions about "
            "movies using graph-enriched context. Your context includes:\n"
            "- Semantic search matches from movie plots\n"
            "- Movie titles and release years\n"
            "- Genres the movie belongs to\n"
            "- Actors who appeared in the movie\n"
            "- Directors of the movie\n\n"
            "When answering, cite the movie title, relevant actors, and genres. "
            "Be specific and reference the enriched graph data."
        ),
        context_providers=[provider],
    )

    session = agent.create_session()

    query = "What are some good science fiction movies and who stars in them?"
    print(f"User: {query}\n")
    print("Answer: ", end="", flush=True)
    response = await agent.run(query, session=session)
    print(response.text)
    print()

User: What are some good science fiction movies and who stars in them?

Answer: From the graph-enriched context, here are some strong science-fiction picks with the main actors and genres:

- Aliens (1986) — Genres: Action, Horror, Adventure, Sci‑Fi. Stars Sigourney Weaver, Carrie Henn, Michael Biehn, Paul Reiser. Director: James Cameron. A militarized rescue/retaliation mission to the colony from Alien (1979).

- The Terminator (1984) — Genres: Thriller, Sci‑Fi, Action. Stars Arnold Schwarzenegger, Paul Winfield, Linda Hamilton, Michael Biehn. Director: James Cameron. A near‑indestructible cyborg is sent from the future to assassinate a woman whose son will lead humanity.

- Star Wars: Episode IV – A New Hope (1977) — Genres: Sci‑Fi, Action, Adventure. Stars Mark Hamill, Harrison Ford, Carrie Fisher, Peter Cushing. Director: George Lucas. A classic space‑opera about rebels, the Death Star, and the fight against the Empire.

- Blade Runner (1982) — Genres: Action, Sci‑Fi, Thriller. Sta

## Experiment

Try modifying your retrieval query:
- Add `node.imdbRating` to the RETURN clause to include ratings
- Change the actor limit from 5 to 3 or 10
- Modify `top_k` to retrieve more or fewer matches

## Summary

Adding a `retrieval_query` to `Neo4jContextProvider` combines vector search with graph traversal. The Cypher query receives matched `node` and `score` variables from the index search, then traverses relationships to collect genres, actors, and directors. The agent gets structured metadata alongside plot text, producing more specific and grounded responses.

**What's next:** The next labs cover fulltext and hybrid search as alternative search modes.